In [ ]:
!pip install -q vllm

In [ ]:
import torch
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-3B-Instruct",
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
)
print(torch.cuda.get_device_name(0))

In [ ]:
import time, numpy as np, pandas as pd

def bench(llm, n_prompts, max_tokens=128, tag=""):
    prompts = ["Summarize the causes of the French Revolution."] * n_prompts
    sp = SamplingParams(max_tokens=max_tokens, temperature=0.0, ignore_eos=True)

    t0 = time.perf_counter()
    outs = llm.generate(prompts, sp)
    elapsed = time.perf_counter() - t0

    total_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    try:
        lat = [o.metrics.finished_time - o.metrics.arrival_time for o in outs]
        lat_mean, lat_p95 = float(np.mean(lat)), float(np.percentile(lat, 95))
    except Exception:
        lat_mean = lat_p95 = elapsed

    return {
        "tag": tag,
        "batch_size": n_prompts,
        "max_tokens": max_tokens,
        "elapsed_s": round(elapsed, 2),
        "output_tokens": total_tokens,
        "tokens_per_s": round(total_tokens / elapsed, 1),
        "requests_per_s": round(n_prompts / elapsed, 2),
        "latency_mean_s": round(lat_mean, 3),
        "latency_p95_s": round(lat_p95, 3),
        "peak_vram_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    }

rows = []
for bs in [1, 2, 4, 8, 16, 32, 64]:
    torch.cuda.reset_peak_memory_stats()
    try:
        row = bench(llm, bs, tag="fp16_util0.85")
        rows.append(row)
        print(row)
    except Exception as e:
        rows.append({"tag": "fp16_util0.85", "batch_size": bs, "error": str(e)[:120]})
        print("FAILED at batch", bs, "->", str(e)[:120])

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/results.csv", index=False)
df

In [ ]:
import matplotlib.pyplot as plt

ok = df[df.get("tokens_per_s").notna()]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ok.batch_size, ok.tokens_per_s, marker="o")
ax[0].set(xlabel="batch size", ylabel="output tokens/sec", title="Throughput vs batch size")
ax[1].plot(ok.batch_size, ok.latency_mean_s, marker="o", color="crimson")
ax[1].set(xlabel="batch size", ylabel="mean latency (s)", title="Latency vs batch size")
plt.tight_layout()
plt.savefig("/kaggle/working/throughput_latency.png", dpi=150)
plt.show()